In [6]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install matplotlib pandas numpy scikit-learn yfinance pandas-ta tqdm seaborn plotly ipywidgets

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.0%2Bcu118-cp311-cp311-win_amd64.whl.metadata (29 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchvision-0.22.0%2Bcu118-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.7.0%2Bcu118-cp311-cp311-win_amd64.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/filelock-3.13.1-py3-none-any.whl.metadata (2.8 kB)
  Using cached https://download.pytorch.org/whl/sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
  Using cached https://download.pytorch.org/whl/networkx-3.3-py3-none-any.whl.metadata (5.1 kB)
  Using cached https://download.pytorch.org/whl/fsspec-2024.6.1-py3-none-any.whl.metadata (11 kB)
  Using cached https://download.pytorch.org/whl/numpy-2.1.2-cp311-cp311-win_amd64.whl.metadata (59 kB)
  Using cached https://download.pytorch.org/whl/pillow-11.0.0-cp311-cp311-win_amd64.w

In [7]:
# Core imports
import os
import random
import warnings

import numpy as np

warnings.filterwarnings('ignore')

# Data visualization
%matplotlib inline

# Machine learning
import torch

In [8]:
TINKOFF_API_PROD = 'invest-public-api.tinkoff.ru:443'
TINKOFF_API_SANDBOX = 'sandbox-invest-public-api.tinkoff.ru:443'

In [10]:
INVEST_API_KEY = input("Enter ML developer Invest API Key: ")

In [11]:
# Check CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
print(f"Current CUDA device: {torch.cuda.current_device()}")
print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.7.0+cu118
CUDA available: True
CUDA device count: 1
Current CUDA device: 0
CUDA device name: NVIDIA GeForce RTX 4080 SUPER


In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [13]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed()

In [14]:
x = torch.randn(3, 3).to(device)
y = torch.randn(3, 3).to(device)
z = x * y
print(z)

tensor([[ 0.0900,  0.0689,  0.1898],
        [ 0.2557,  1.8974,  0.1843],
        [ 2.1154, -0.8435,  0.3773]], device='cuda:0')


In [ ]:


from grpc import ssl_channel_credentials
from grpc import aio

from externalClients.TInvestApi.proto import (
    marketdata_pb2_grpc
)

channel = aio.secure_channel(TINKOFF_API_PROD, ssl_channel_credentials())
api_key = INVEST_API_KEY

def get_metadata(self):
    return [('authorization', f'Bearer {self.api_key}')]

async def close():
    await channel.close()



stub = marketdata_pb2_grpc.MarketDataServiceStub(channel)

await stub.GetCandles()

In [21]:
import asyncio
from datetime import datetime, timedelta
from typing import Optional, List, Dict, Union
import pandas as pd
from grpc import aio, ssl_channel_credentials
from google.protobuf.timestamp_pb2 import Timestamp
from externalClients.TInvestApi.proto import marketdata_pb2_grpc
from externalClients.TInvestApi.proto.marketdata_pb2 import (
    GetCandlesRequest,
    GetCandlesResponse,
    HistoricCandle,
    CandleInterval,
    MarketDataRequest,
    MarketDataResponse,
    SubscribeCandlesRequest,
    SubscriptionAction,
    SubscriptionInterval
)

class TinkoffDataProvider:
    def __init__(self, api_key: str, prod_endpoint: str = "invest-public-api.tinkoff.ru:443"):
        """
        Initialize Tinkoff Invest API data provider.

        Args:
            api_key: Your Tinkoff Invest API token
            prod_endpoint: API endpoint (default is production)
        """
        self.api_key = api_key
        self.endpoint = prod_endpoint
        self.channel = None
        self.stub = None
        self._initialize_connection()

    def _initialize_connection(self):
        """Initialize gRPC connection and stub."""
        credentials = ssl_channel_credentials()
        self.channel = aio.secure_channel(self.endpoint, credentials)
        self.stub = marketdata_pb2_grpc.MarketDataServiceStub(self.channel)

    def _get_metadata(self):
        """Get authorization metadata for gRPC calls."""
        return [('authorization', f'Bearer {self.api_key}')]

    async def close(self):
        """Close the gRPC channel."""
        if self.channel:
            await self.channel.close()

    def _convert_quotation_to_float(self, quotation) -> float:
        """Convert Tinkoff Quotation type to float."""
        return float(quotation.units + quotation.nano / 1e9)

    def _convert_timestamp_to_datetime(self, timestamp) -> datetime:
        """Convert protobuf Timestamp to Python datetime."""
        return datetime.utcfromtimestamp(timestamp.seconds + timestamp.nanos / 1e9)

    async def get_historical_candles(
        self,
        instrument_id: str,
        from_time: datetime,
        to_time: datetime,
        interval: CandleInterval,
        candle_source_type: Optional[int] = None,
        limit: Optional[int] = None
    ) -> pd.DataFrame:
        """
        Get historical candles for specified instrument and time range.

        Args:
            instrument_id: FIGI or instrument_uid
            from_time: Start time of the period
            to_time: End time of the period
            interval: Candle interval (see CandleInterval enum)
            candle_source_type: Optional candle source type
            limit: Optional maximum number of candles

        Returns:
            Pandas DataFrame with historical candles
        """
        # Convert datetime to protobuf Timestamp
        to_ts = Timestamp()
        to_ts.FromDatetime(to_time)

        # Create request
        request = GetCandlesRequest(
            instrument_id=instrument_id,
            to=to_ts,
            interval=interval,
            candle_source_type=candle_source_type,
            limit=limit
        )
        setattr(request, "from", from_time)

        # Make API call
        response: GetCandlesResponse = await self.stub.GetCandles(
            request,
            metadata=self._get_metadata()
        )

        # Process candles into DataFrame
        candles = []
        for candle in response.candles:
            candles.append({
                'time': self._convert_timestamp_to_datetime(candle.time),
                'open': self._convert_quotation_to_float(candle.open),
                'high': self._convert_quotation_to_float(candle.high),
                'low': self._convert_quotation_to_float(candle.low),
                'close': self._convert_quotation_to_float(candle.close),
                'volume': candle.volume,
                'is_complete': candle.is_complete,
                'source': candle.candle_source
            })

        return pd.DataFrame(candles).set_index('time')

    async def subscribe_to_candles(
        self,
        instrument_ids: List[str],
        interval: SubscriptionInterval,
        callback: callable
    ) -> None:
        """
        Subscribe to real-time candle updates.

        Args:
            instrument_ids: List of instrument IDs to subscribe to
            interval: Subscription interval (see SubscriptionInterval enum)
            callback: Function to call when new data arrives
        """
        # Create subscription requests
        subscription_list = [
            SubscribeCandlesRequest(
                subscription_action=SubscriptionAction.SUBSCRIPTION_ACTION_SUBSCRIBE,
                instruments=[
                    SubscribeCandlesRequest.CandleInstrument(
                        instrument_id=inst_id,
                        interval=interval
                    )
                    for inst_id in instrument_ids
                ]
            )
        ]

        # Create market data request
        request = MarketDataRequest(
            subscribe_candles_request=subscription_list
        )

        # Start streaming
        async for response in self.stub.MarketDataStream(
            iter([request]),
            metadata=self._get_metadata()
        ):
            if response.HasField('candle'):
                candle_data = {
                    'time': self._convert_timestamp_to_datetime(response.candle.time),
                    'open': self._convert_quotation_to_float(response.candle.open),
                    'high': self._convert_quotation_to_float(response.candle.high),
                    'low': self._convert_quotation_to_float(response.candle.low),
                    'close': self._convert_quotation_to_float(response.candle.close),
                    'volume': response.candle.volume,
                    'is_complete': response.candle.is_complete,
                    'instrument_id': response.candle.figi
                }
                await callback(candle_data)

    async def get_last_n_candles(
        self,
        instrument_id: str,
        interval: CandleInterval,
        n_candles: int,
        candle_source_type: Optional[int] = None
    ) -> pd.DataFrame:
        """
        Get last N candles for specified instrument.

        Args:
            instrument_id: FIGI or instrument_uid
            interval: Candle interval
            n_candles: Number of candles to retrieve
            candle_source_type: Optional candle source type

        Returns:
            Pandas DataFrame with last N candles
        """
        # Calculate time range based on interval
        now = datetime.utcnow()

        # Estimate time range needed (approximate)
        interval_mapping = {
            CandleInterval.CANDLE_INTERVAL_1_MIN: timedelta(minutes=n_candles),
            CandleInterval.CANDLE_INTERVAL_5_MIN: timedelta(minutes=5*n_candles),
            CandleInterval.CANDLE_INTERVAL_15_MIN: timedelta(minutes=15*n_candles),
            CandleInterval.CANDLE_INTERVAL_HOUR: timedelta(hours=n_candles),
            CandleInterval.CANDLE_INTERVAL_DAY: timedelta(days=n_candles),
        }

        from_time = now - interval_mapping.get(interval, timedelta(days=n_candles))

        return await self.get_historical_candles(
            instrument_id=instrument_id,
            from_time=from_time,
            to_time=now,
            interval=interval,
            candle_source_type=candle_source_type,
            limit=n_candles
        )


In [25]:
provider = TinkoffDataProvider(api_key=INVEST_API_KEY)

try:
    # Get historical data
    df = await provider.get_historical_candles(
        instrument_id="e6123145-9665-43e0-8413-cd61b8aa9b13",
        from_time=datetime.utcnow() - timedelta(hours=40),
        to_time=datetime.utcnow(),
        interval=CandleInterval.CANDLE_INTERVAL_1_MIN
    )
    print(df.head())
    print(df.shape)

finally:
    await provider.close()


                       open    high     low   close  volume  is_complete  \
time                                                                       
2025-04-22 08:59:00  308.78  308.79  308.70  308.71    1899         True   
2025-04-22 09:00:00  308.70  308.70  308.27  308.46    1604         True   
2025-04-22 09:01:00  308.43  308.43  308.17  308.23    2583         True   
2025-04-22 09:02:00  308.20  308.49  308.15  308.44    3681         True   
2025-04-22 09:03:00  308.49  308.62  308.46  308.49    1524         True   

                     source  
time                         
2025-04-22 08:59:00       1  
2025-04-22 09:00:00       1  
2025-04-22 09:01:00       1  
2025-04-22 09:02:00       1  
2025-04-22 09:03:00       1  
(1684, 7)
